# Size the pipeline to this machine, on synthetic cells

Open this notebook from its GitHub link on a Colab L4 runtime, the experiment's one GPU, at the
latest runtime version (Runtime > Change runtime type): <https://colab.research.google.com/github/meiazero/garimpo-uncertainty/blob/main/notebooks/03_tune_hardware.ipynb>

The aim is a run that uses the machine usefully, not the fastest run. Each section runs one hot
path of the pipeline, measures how busy the GPU, the CPUs and the disk are meanwhile, names what
holds it back, and finds the smallest setting that lifts that limit. The verdict also says how long
the full run's training and searches take on this machine.

Nothing here touches the real data or Drive: the cells are synthetic, of the real shape (6 bands,
512 x 512 pixels), and the models have random weights. Times per decoder call and per update hold
for this hardware; how many calls a search makes differs for a trained model, so the projected
hours are estimates. Every candidate is checked against a reference to the bit, and only one that
keeps every bit counts as safe; one that moves a number is reported for a decision.

`requirements.txt` pins the versions the latest Colab runtime publishes. An older runtime version
ships another torch, 2.11.0 on 2026.10, so the setup stops there before installing anything.

1. Set the form in section 1.
2. Run Runtime > Run all.
3. Read the verdict at the end of section 3: `TUNING: PASS` or `TUNING: FAIL`, then the findings.

| Section | What it does |
|---|---|
| 1. Form | What to measure |
| 2. Setup | Clones the branch and installs it, keeping Colab's torch and CUDA |
| 3. Tests and tuning | The tests when asked, then `garimpo tune`: every measurement and the verdict |
| 4. Release | Releases the GPU, so the session stops spending compute units |

`garimpo tune` measures in this notebook's Python, under torch's deterministic algorithms, as every
`garimpo` command runs: an op without one stops its section, and a training, batch or precision run
whose two runs differ fails the verdict. The measurements live in the package, in
`garimpo_uncertainty/tuning/`, so the notebook only passes the form to the command.

## 1. Form

- `BRANCH`: the code to measure.
- `ARCHS`: the architectures, by registered name; empty is every one.
- `CELLS`: synthetic cells per search, batch and precision measurement, a multiple of every batch size.
- `SEARCH_UNET_LAYERS`: the U-Net layers the search times; empty is every layer.
- `TIMED_UNET_LAYERS`: the U-Net layers the batch and precision measurements time. `decoder.blocks.0` is the slowest
  layer of the pilot and `decoder.blocks.4` the one it chose.
- `LOADER_CELLS`: the cells of the synthetic store that data loading and training read from the disk.
- `WORKERS`: data-loader worker counts, or `auto` for 0, 2, 4, half and all of the CPUs.
- `BATCH_SIZES`, `STEPS`, `REPEATS`: the eval batch sizes, the updates per timed training run, and the
  timed runs per candidate.
- `RUN_TESTS`: run the test suite on the GPU before the tuning.
- `RELEASE`: release the GPU after the verdict.

In [ ]:
# @title The form
BRANCH = "main"  # @param {type:"string"}
ARCHS = ""  # @param {type:"string"}
CELLS = 64  # @param {type:"integer"}
SEARCH_UNET_LAYERS = ""  # @param {type:"string"}
TIMED_UNET_LAYERS = "decoder.blocks.0 decoder.blocks.4"  # @param {type:"string"}
LOADER_CELLS = 256  # @param {type:"integer"}
WORKERS = "auto"  # @param {type:"string"}
BATCH_SIZES = "16 32 64"  # @param {type:"string"}
STEPS = 20  # @param {type:"integer"}
REPEATS = 3  # @param {type:"integer"}
RUN_TESTS = False  # @param {type:"boolean"}
RELEASE = True  # @param {type:"boolean"}

ARCH_CHOICE = f"--archs {ARCHS}" if ARCHS.strip() else ""
WORKER_COUNTS = "" if WORKERS.strip() == "auto" else f"--workers {WORKERS}"

## 2. Setup

The runtime's disk is erased when a session ends, so this cell clones the code again, or moves the
clone already there to the last commit of `BRANCH`, and installs `requirements.txt`. The install
stops, and names the package, when pip would replace Colab's torch or a CUDA library, which pick
the GPU's kernels: choose the latest runtime version then. Every later cell calls a `garimpo`
command, which loads the code just installed. When a step fails, `SETUP_OK` is false and
the cells that run the code skip, so none runs an older install; the release still runs.

In [ ]:
# @title Setup: clone BRANCH and install it
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone; erased with the session
steps = [] if Path(CODE, ".git").is_dir() else [["git", "clone", "-q", REPO, CODE]]
steps += [
    ["git", "-C", CODE, "fetch", "-q", "origin", BRANCH],
    ["git", "-C", CODE, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"],
    ["git", "-C", CODE, "submodule", "update", "-q", "--init", "--recursive"],
    [sys.executable, f"{CODE}/scripts/colab_install.py"],
    ["git", "-C", CODE, "log", "-1", "--format=Running commit %h %cs %s"],
]


def ran(step: list[str]) -> bool:
    """Run one step inside the clone once it exists, as pip reads requirements.txt's -e . there."""
    try:
        done = subprocess.run(step, check=False, cwd=CODE if Path(CODE).is_dir() else None)  # noqa: S603
    except OSError as error:  # a raise here would cancel the release
        print(f"Could not start {step[0]}: {error}")
        return False
    return done.returncode == 0


# all() stops at the first failure; a later cell then skips rather than run an older install.
SETUP_OK = all(ran(step) for step in steps)
if Path(CODE).is_dir():
    os.chdir(CODE)
if not SETUP_OK:
    print("The setup failed, so the cells that run the code skip: read the error above.")

## 3. Tests and tuning

When `RUN_TESTS` is checked, the test suite runs first on CUDA, so a failing test shows before any
timing. Then `garimpo tune` runs every section on synthetic cells of the real shape, each timing
the code `garimpo` runs: the hardware the session got, data loading at each worker count, training
at each worker count, the searches with and without `Memo`, `compass_scores` at each batch size and
under four sets of precision flags, and LightGBM's density ratio at each thread count. A section
that fails is reported and the rest still run. The verdict at the end names every finding and every
failure: a missing section, a run that gave other bits when repeated or at another worker
count, or a loop starved of data; `TUNING: PASS` only without one. A faster candidate with
other bits than strict float32 is a finding to decide on.
The rows are kept in `outputs/tuning/results.json`.

In [ ]:
# @title Run the tests when the form asks, then the tuning
if not SETUP_OK:
    print("Skipped: the setup failed.")
else:
    if RUN_TESTS:
        !GARIMPO_TEST_DEVICE=cuda python -m pytest -q
    !garimpo tune {ARCH_CHOICE} --cells {CELLS} --search-unet-layers {SEARCH_UNET_LAYERS} --timed-unet-layers {TIMED_UNET_LAYERS} --loader-cells {LOADER_CELLS} {WORKER_COUNTS} --batch-sizes {BATCH_SIZES} --steps {STEPS} --repeats {REPEATS}

## 4. Release

Releases the GPU when `RELEASE` is checked, which ends the session and every result with it, so
copy the verdict above first.

In [ ]:
# @title Release the runtime
import os

from google.colab import drive, runtime

if RELEASE:
    if os.path.ismount("/content/drive"):
        try:
            drive.flush_and_unmount()
        except Exception as error:  # noqa: BLE001 -- the runtime must be released whatever failed
            print(f"Drive did not confirm the last writes; check the files in Drive: {error}")
    runtime.unassign()
else:
    print("Kept: RELEASE is not checked.")